# Experiment 4.0.5 — Temporal resolution vs event capacity

Analysis-only notebook. Training is performed by the 45-task Slurm array. This notebook reads finalized artifacts only.

Primary comparisons:
- `repeat4 - fixed250`: more internal SNN timesteps without adding within-bin information.
- `raw64 - repeat4`: value of genuine within-bin event timing.
- Binary vs Multi-H vs Multi-HO across each representation.
- Output WholeCount vs Hidden WholeCount + Linear vs Uend + Linear.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
root = repo_root / 'notebooks' / 'artifacts' / 'experiment_4_0_5_temporal_resolution_event_capacity' / 'hierarchical_temporal_resolution_event_cap_v1'
runs_file = root / 'runs.csv'
summary_file = root / 'summary.csv'
effects_file = root / 'paired_effects.csv'
print('Artifacts:', root)
print('runs.csv exists:', runs_file.exists())

Artifacts: /home/zhaolongwei_umass_edu/projects/writingRing/notebooks/artifacts/experiment_4_0_5_temporal_resolution_event_capacity/hierarchical_temporal_resolution_event_cap_v1
runs.csv exists: True


In [2]:
if not runs_file.exists():
    print('Finalized artifacts are not available yet. Run the Slurm array and finalizer first.')
else:
    runs = pd.read_csv(runs_file)
    test = runs[runs['split'] == 'test'].copy()
    display(test[['representation','variant','seed','valid_count_balanced_accuracy','hidden_count_linear_balanced_accuracy','uend_linear_balanced_accuracy']].sort_values(['representation','variant','seed']))

,representation,variant,seed,valid_count_balanced_accuracy,hidden_count_linear_balanced_accuracy,uend_linear_balanced_accuracy
2,fixed250,binary,11,0.352096,0.310429,0.375182
5,fixed250,binary,23,0.323809,0.363130,0.357116
8,fixed250,binary,37,0.299874,0.327715,0.364395
11,fixed250,binary,53,0.287091,0.286335,0.344268
14,fixed250,binary,71,0.365588,0.393896,0.450892
17,fixed250,multi_h,11,0.295050,0.406790,0.396297
20,fixed250,multi_h,23,0.290553,0.347438,0.370544
23,fixed250,multi_h,37,0.132323,0.287201,0.369548
26,fixed250,multi_h,53,0.188585,0.384713,0.382753
29,fixed250,multi_h,71,0.314857,0.342513,0.376636


In [3]:
if summary_file.exists():
    summary = pd.read_csv(summary_file)
    display(summary)

,representation,variant,hidden_cap,output_cap,valid_count_balanced_accuracy_mean,valid_count_balanced_accuracy_std,hidden_count_linear_balanced_accuracy_mean,hidden_count_linear_balanced_accuracy_std,uend_linear_balanced_accuracy_mean,uend_linear_balanced_accuracy_std,...,output_tail_event_fraction_mean,output_tail_event_fraction_std,state_mean_events_per_neuron_second_mean,state_mean_events_per_neuron_second_std,output_mean_events_per_neuron_second_mean,output_mean_events_per_neuron_second_std,state_fraction_at_cap_mean,state_fraction_at_cap_std,output_fraction_at_cap_mean,output_fraction_at_cap_std
0,fixed250,binary,1,1,0.325692,0.033350,0.336301,0.042651,0.378371,0.042070,...,0.349344,0.135506,0.981213,0.054382,0.556244,0.078101,0.245303,0.013595,0.139061,0.019525
1,fixed250,multi_h,31,1,0.244274,0.079587,0.353731,0.045735,0.379156,0.010950,...,0.408202,0.378342,8.800477,6.172617,0.297384,0.203083,0.018693,0.031727,0.074346,0.050771
2,fixed250,multi_ho,31,31,0.437105,0.033605,0.472988,0.029186,0.501342,0.013664,...,0.784098,0.022412,25.057656,1.932214,22.074192,5.372376,0.065588,0.012072,0.042367,0.005606
3,raw64,binary,1,1,0.201907,0.029776,0.215023,0.017745,0.222105,0.028766,...,0.006300,0.003111,1.574630,0.092197,1.879180,0.516621,0.024604,0.001441,0.029362,0.008072
4,raw64,multi_h,31,1,0.190462,0.035924,0.208307,0.038350,0.215338,0.016177,...,0.099082,0.212360,2.968860,3.280795,1.363176,0.916552,0.000230,0.000515,0.021300,0.014321
5,raw64,multi_ho,31,31,0.289547,0.037988,0.306097,0.013581,0.336187,0.038998,...,0.172020,0.097150,23.596727,7.196228,115.869366,46.722167,0.000003,0.000005,0.000731,0.000707
6,repeat4,binary,1,1,0.135133,0.040116,0.155490,0.010580,0.176793,0.034359,...,0.022172,0.007068,0.706033,0.188910,0.408240,0.114177,0.044127,0.011807,0.025515,0.007136
7,repeat4,multi_h,31,1,0.131961,0.027061,0.165725,0.032488,0.179330,0.029040,...,0.005837,0.006617,30.391679,12.104205,0.246806,0.050255,0.036945,0.018323,0.015425,0.003141
8,repeat4,multi_ho,31,31,0.222219,0.017804,0.215870,0.028130,0.233931,0.012153,...,0.796141,0.029644,65.336443,31.182265,157.282633,43.644552,0.076540,0.063603,0.229718,0.102312


In [4]:
if runs_file.exists():
    test = pd.read_csv(runs_file).query("split == 'test'")
    plot_rows = []
    for (representation, variant), group in test.groupby(['representation','variant']):
        for readout, column in {
            'Output WholeCount': 'valid_count_balanced_accuracy',
            'Hidden Count + Linear': 'hidden_count_linear_balanced_accuracy',
            'Uend + Linear': 'uend_linear_balanced_accuracy',
        }.items():
            plot_rows.append({
                'representation': representation,
                'variant': variant,
                'readout': readout,
                'mean_ba': group[column].mean(),
                'std_ba': group[column].std(),
            })
    plot_df = pd.DataFrame(plot_rows)
    display(plot_df)

,representation,variant,readout,mean_ba,std_ba
0,fixed250,binary,Output WholeCount,0.325692,0.033350
1,fixed250,binary,Hidden Count + Linear,0.336301,0.042651
2,fixed250,binary,Uend + Linear,0.378371,0.042070
3,fixed250,multi_h,Output WholeCount,0.244274,0.079587
4,fixed250,multi_h,Hidden Count + Linear,0.353731,0.045735
5,fixed250,multi_h,Uend + Linear,0.379156,0.010950
6,fixed250,multi_ho,Output WholeCount,0.437105,0.033605
7,fixed250,multi_ho,Hidden Count + Linear,0.472988,0.029186
8,fixed250,multi_ho,Uend + Linear,0.501342,0.013664
9,raw64,binary,Output WholeCount,0.201907,0.029776


In [5]:
if effects_file.exists():
    effects = pd.read_csv(effects_file)
    display(effects.groupby(['effect','readout'], dropna=False)['delta_balanced_accuracy'].agg(['mean','std','count']).reset_index())

,effect,readout,mean,std,count
0,multi_h_minus_binary,hidden_whole_count_linear,0.006983,0.047519,15
1,multi_h_minus_binary,output_whole_count,-0.032012,0.054998,15
2,multi_h_minus_binary,uend_linear,-0.001148,0.040187,15
3,multi_ho_minus_binary,hidden_whole_count_linear,0.096047,0.043489,15
4,multi_ho_minus_binary,output_whole_count,0.095379,0.048646,15
5,multi_ho_minus_binary,uend_linear,0.098064,0.043704,15
6,raw64_minus_fixed250,hidden_whole_count_linear,-0.144531,0.050892,15
7,raw64_minus_fixed250,output_whole_count,-0.108384,0.061237,15
8,raw64_minus_fixed250,uend_linear,-0.161746,0.045092,15
9,raw64_minus_repeat4,hidden_whole_count_linear,0.064114,0.043436,15


## Interpretation guide

1. If `repeat4 > fixed250`, more binary/multi-event firing opportunities help even without adding information.
2. If `raw64 > repeat4`, genuine within-bin timing adds discriminative information.
3. If `Multi-HO - Binary` shrinks from Fixed250 to Repeat4/Raw64, multi-event communication is partly compensating for coarse temporal discretization.
4. If `Uend + Linear` stays high while Output WholeCount improves with temporal resolution, the main gain is better state-to-spike export rather than better endpoint memory.